In [1]:
# Everything we'll need for this analysis in one place
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# A cleaner default look for all the charts
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)  # show every column when printing a DataFrame

In [2]:
# These CSVs come straight from the BigQuery views,
# so the cleaning and derived metrics are already done.
df      = pd.read_csv("v_nuclear.csv")
europe  = pd.read_csv("v_europe_yearly.csv")
changes = pd.read_csv("v_country_change.csv")

# Quick check that everything loaded as expected
print(df.shape)       # should be (224, 13)
print(europe.shape)   # should be (16, 4)
print(changes.shape)  # should be (14, 6)

df.head()

(224, 12)
(16, 4)
(14, 6)


,Country,Year,Cluster_Type,Installed_Capacity_MW,Nuclear_Generation_TWh,Nuclear_Share_Pct,capacity_factor,implied_total_twh,yoy_change_twh,yoy_change_pct,generation_rank,is_inconsistent
0,Belgium,2010,Moderate Producer,5900,43.68,48.0,0.845,91.0,NaN,NaN,5,False
1,Belgium,2011,Moderate Producer,5900,39.23,45.5,0.759,86.2,-4.45,-0.102,5,False
2,Belgium,2012,Moderate Producer,5900,39.13,48.2,0.757,81.2,-0.10,-0.003,5,False
3,Belgium,2013,Moderate Producer,5900,40.59,47.4,0.785,85.6,1.46,0.037,5,False
4,Belgium,2014,Moderate Producer,5900,37.56,48.7,0.727,77.1,-3.03,-0.075,5,False


In [3]:
# Boil each country down to a handful of numbers that describe its nuclear profile.
features = df.groupby("Country").agg(
    cluster_type=("Cluster_Type", "first"),
    avg_share=("Nuclear_Share_Pct", "mean"),          # how much the country relies on nuclear
    share_volatility=("Nuclear_Share_Pct", "std"),    # how much that share moves around
    avg_generation=("Nuclear_Generation_TWh", "mean"),# how big the nuclear fleet is
    avg_capacity_factor=("capacity_factor", "mean"),  # how hard the reactors are run
)

# Trend: the slope of a straight line through the share over time,
# i.e. how many percentage points the share gains or loses per year.
features["share_trend"] = df.groupby("Country").apply(
    lambda g: np.polyfit(g["Year"], g["Nuclear_Share_Pct"], 1)[0]
)

features.round(2).sort_values("avg_share", ascending=False)

,cluster_type,avg_share,share_volatility,avg_generation,avg_capacity_factor,share_trend
Country,,,,,,
France,Major Producer,70.01,4.98,419.64,0.76,-0.35
Slovakia,Small / Niche Producer,53.84,4.75,13.41,0.77,0.61
Belgium,Moderate Producer,48.42,1.35,39.47,0.76,0.18
Hungary,Small / Niche Producer,48.39,1.26,13.91,0.79,-0.01
Bulgaria,Small / Niche Producer,39.08,1.47,14.06,0.80,-0.03
Sweden,Moderate Producer,37.86,1.58,62.70,0.80,0.04
Slovenia,Small / Niche Producer,37.17,1.59,4.88,0.80,0.00
Czech Republic,Moderate Producer,35.52,1.45,29.71,0.79,0.00
Switzerland,Small / Niche Producer,35.45,1.88,20.63,0.78,0.01


In [7]:
# =====================================================
# Clustering: do the countries naturally fall into groups?
# =====================================================
import warnings
from sklearn.metrics import silhouette_score

# Hide the harmless KMeans memory-leak warning that shows up on Windows
warnings.filterwarnings("ignore", message="KMeans is known to have a memory leak")


# --- 1. Pick the features -----------------------------------------------
# We describe each country with three things:
#   - how much it relies on nuclear (average share)
#   - where it's heading (share trend, points per year)
#   - how big its fleet is (average generation)
# France produces ten times more than anyone else, so we use the log of
# generation; otherwise France alone would decide every cluster.
X = pd.DataFrame({
    "avg_share":      features["avg_share"],
    "share_trend":    features["share_trend"],
    "log_generation": np.log1p(features["avg_generation"]),
})

# Put all three on the same scale so no feature wins just because of its units
X_scaled = StandardScaler().fit_transform(X)


# --- 2. How many clusters? ----------------------------------------------
# Silhouette goes from -1 to 1: the higher, the cleaner the separation.
scores = []
for k in range(2, 6):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    scores.append({"clusters": k, "silhouette": round(silhouette_score(X_scaled, labels), 3)})

print("How well do the countries separate for each number of clusters?")
display(pd.DataFrame(scores))


# --- 3. Run KMeans with 3 clusters --------------------------------------
# We go with 3 so we can compare against the 3 categories in Cluster_Type.
features["kmeans_cluster"] = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_scaled)

# KMeans numbers the clusters at random (0, 1, 2), so we give them real names.
# We look up which cluster France and Germany landed in, so this works
# whatever numbers KMeans happens to pick.
france_id  = features.loc["France", "kmeans_cluster"]
germany_id = features.loc["Germany", "kmeans_cluster"]

features["kmeans_label"] = np.select(
    [features["kmeans_cluster"] == france_id,
     features["kmeans_cluster"] == germany_id],
    ["Nuclear giant", "Phasing out"],
    default="Stable mid-size",
)


# --- 4. Who ended up where? ---------------------------------------------
print("\nCountries in each cluster:")
for label, group in features.groupby("kmeans_label"):
    print(f"  {label}: {', '.join(group.index)}")


# --- 5. Compare with the dataset's own labels ---------------------------
# Side by side: how the dataset classifies each country vs. what the data says.
comparison = features[["cluster_type", "kmeans_label", "avg_share", "share_trend", "avg_generation"]] \
    .round(2) \
    .sort_values(["kmeans_label", "avg_generation"], ascending=[True, False]) \
    .rename(columns={
        "cluster_type":   "Dataset label",
        "kmeans_label":   "KMeans group",
        "avg_share":      "Avg share (%)",
        "share_trend":    "Trend (pts/year)",
        "avg_generation": "Avg generation (TWh)",
    })

print("\nDataset label vs. KMeans group:")
display(comparison)

How well do the countries separate for each number of clusters?


,clusters,silhouette
0,2,0.483
1,3,0.454
2,4,0.288
3,5,0.275



Countries in each cluster:
  Nuclear giant: France
  Phasing out: Germany
  Stable mid-size: Belgium, Bulgaria, Czech Republic, Finland, Hungary, Netherlands, Romania, Slovakia, Slovenia, Spain, Sweden, Switzerland

Dataset label vs. KMeans group:


,Dataset label,KMeans group,Avg share (%),Trend (pts/year),Avg generation (TWh)
Country,,,,,
France,Major Producer,Nuclear giant,70.01,-0.35,419.64
Germany,Major Producer,Phasing out,10.08,-1.67,63.29
Sweden,Moderate Producer,Stable mid-size,37.86,0.04,62.70
Spain,Moderate Producer,Stable mid-size,20.04,-0.02,48.97
Belgium,Moderate Producer,Stable mid-size,48.42,0.18,39.47
Czech Republic,Moderate Producer,Stable mid-size,35.52,0.00,29.71
Switzerland,Small / Niche Producer,Stable mid-size,35.45,0.01,20.63
Finland,Small / Niche Producer,Stable mid-size,35.22,0.83,20.32
Bulgaria,Small / Niche Producer,Stable mid-size,39.08,-0.03,14.06


In [8]:
# =====================================================
# Segmentation: a simpler, rule-based way to group countries
# =====================================================
# Clustering told us the data has two outliers and one big group, which
# isn't very useful for a dashboard. So we build our own segments from two
# questions anyone can understand: how much does the country rely on
# nuclear, and is that going up, down or holding steady?

# How big a slice of the country's electricity comes from nuclear
features["dependency"] = pd.cut(
    features["avg_share"],
    bins=[0, 20, 40, 100],
    labels=["Low (<20%)", "Medium (20-40%)", "High (>40%)"],
)

# Is nuclear growing, holding steady or being phased out?
# Half a percentage point a year is our line between "moving" and "stable".
features["trajectory"] = np.select(
    [features["share_trend"] > 0.5, features["share_trend"] < -0.5],
    ["Growing", "Declining"],
    default="Stable",
)

# Show who ends up where
print("Countries by trajectory and dependency:")
for (trajectory, dependency), group in features.groupby(["trajectory", "dependency"], observed=True):
    print(f"  {trajectory} / {dependency}: {', '.join(group.index)}")

Countries by trajectory and dependency:
  Declining / Low (<20%): Germany
  Growing / Medium (20-40%): Finland
  Growing / High (>40%): Slovakia
  Stable / Low (<20%): Netherlands, Romania
  Stable / Medium (20-40%): Bulgaria, Czech Republic, Slovenia, Spain, Sweden, Switzerland
  Stable / High (>40%): Belgium, France, Hungary


In [9]:
# =====================================================
# Export the country-level results for Power BI
# =====================================================
# One row per country with all the indicators, the KMeans group and
# the segments. Power BI will join this to the other tables on Country.
export = features.reset_index()[[
    "Country",
    "cluster_type",
    "avg_share",
    "share_volatility",
    "share_trend",
    "avg_generation",
    "avg_capacity_factor",
    "kmeans_label",
    "dependency",
    "trajectory",
]].round(3)

export.to_csv("country_features.csv", index=False)

print(f"Saved country_features.csv with {len(export)} countries")
export

Saved country_features.csv with 14 countries


,Country,cluster_type,avg_share,share_volatility,share_trend,avg_generation,avg_capacity_factor,kmeans_label,dependency,trajectory
0,Belgium,Moderate Producer,48.419,1.350,0.178,39.467,0.764,Stable mid-size,High (>40%),Stable
1,Bulgaria,Small / Niche Producer,39.081,1.468,-0.026,14.058,0.802,Stable mid-size,Medium (20-40%),Stable
2,Czech Republic,Moderate Producer,35.519,1.454,0.005,29.707,0.789,Stable mid-size,Medium (20-40%),Stable
3,Finland,Small / Niche Producer,35.225,5.725,0.832,20.318,0.767,Stable mid-size,Medium (20-40%),Growing
4,France,Major Producer,70.006,4.978,-0.345,419.641,0.760,Nuclear giant,High (>40%),Stable
5,Germany,Major Producer,10.081,8.144,-1.668,63.291,0.785,Phasing out,Low (<20%),Declining
6,Hungary,Small / Niche Producer,48.394,1.259,-0.014,13.912,0.794,Stable mid-size,High (>40%),Stable
7,Netherlands,Small / Niche Producer,3.131,1.258,0.001,3.368,0.801,Stable mid-size,Low (<20%),Stable
8,Romania,Small / Niche Producer,19.400,1.308,0.002,8.909,0.782,Stable mid-size,Low (<20%),Stable
9,Slovakia,Small / Niche Producer,53.838,4.750,0.613,13.411,0.770,Stable mid-size,High (>40%),Growing
